 Install Java & PySpark

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark findspark -q

Start a Spark session

In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SkyPredict") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Spark initialized.")

Spark version: 4.0.2
Spark is ready!


In [ ]:
# Install other libraries

!pip install pymongo requests pandas \
     matplotlib plotly streamlit -q

print("Libraries installed.")

    1.7/1.7 MB 21.9 MB/s eta 0:00:00
    9.1/9.1 MB 79.9 MB/s eta 0:00:00
    331.1/331.1 kB 17.4 MB/s eta 0:00:00
    6.9/6.9 MB 88.9 MB/s eta 0:00:00
All libraries installed!


In [ ]:
# quick check that spark is working

data = [("SkyPredict", 1), ("Spark", 2)]
df = spark.createDataFrame(data, ["name", "id"])
df.show()
print("Setup done.")

+----------+---+
|      name| id|
+----------+---+
|SkyPredict|  1|
|     Spark|  2|
+----------+---+

Setup complete! Ready for Step 2.


In [ ]:
# BTS flight delay data

import requests, zipfile, io, os
import pandas as pd

os.makedirs("/content/data/bts", exist_ok=True)

# BTS download URL pattern (2019-2023, 5 years ~10GB)
years = [2019, 2020, 2021, 2022, 2023]
months = range(1, 13)

for year in years:
    for month in months:
        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"
        print(f"Downloading {year}-{month:02d}...")
        try:
            r = requests.get(url, timeout=60)
            z = zipfile.ZipFile(io.BytesIO(r.content))
            z.extractall(f"/content/data/bts/")
        except Exception as e:
            print(f"  Skipped {year}-{month}: {e}")

print("BTS download finished.")

BTS download complete!


In [ ]:
# OpenSky network data

import requests, os
import pandas as pd

os.makedirs("/content/data/opensky", exist_ok=True)

# Direct Zenodo links - these work from Colab
urls = [
    "https://zenodo.org/record/7923702/files/flightlist_20190101_20190131.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190201_20190228.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190301_20190331.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20200101_20200131.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20200201_20200229.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20200301_20200331.csv.gz",
]

for url in urls:
    filename = url.split("/")[-1]
    filepath = f"/content/data/opensky/{filename}"
    print(f"Downloading {filename}...")
    try:
        r = requests.get(url, timeout=180)
        with open(filepath, "wb") as f:
            f.write(r.content)
        size = os.path.getsize(filepath) / (1024**2)
        print(f"   Done — {size:.1f} MB")
    except Exception as e:
        print(f"   Failed: {e}")

print("OpenSky download finished.")

   Done — 142.7 MB
   Done — 133.4 MB
   Done — 151.7 MB
   Done — 184.9 MB
   Done — 177.7 MB
   Done — 144.6 MB
OpenSky download complete!


In [ ]:
import os

def get_folder_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

bts_size = get_folder_size("/content/data/bts")
noaa_size = get_folder_size("/content/data/noaa")
opensky_size = get_folder_size("/content/data/opensky")

total = bts_size + noaa_size + opensky_size

print(f" BTS Data:     {bts_size / (1024**3):.2f} GB")
print(f" NOAA Data:    {noaa_size / (1024**3):.2f} GB")
print(f" OpenSky Data: {opensky_size / (1024**3):.2f} GB")
print(f"")
print(f" TOTAL:        {total / (1024**3):.2f} GB")

 BTS Data:     13.30 GB
 NOAA Data:    0.00 GB
 OpenSky Data: 0.91 GB

 TOTAL:        14.21 GB


In [ ]:
# NOAA weather data

import requests, os

os.makedirs("/content/data/noaa", exist_ok=True)

years = [2019, 2020, 2021, 2022, 2023]

for year in years:
    url = f"https://noaa-ghcn-pds.s3.amazonaws.com/csv/by_year/{year}.csv"
    filepath = f"/content/data/noaa/ghcn_{year}.csv"
    print(f"Downloading NOAA {year}...")
    try:
        r = requests.get(url, timeout=300, stream=True)
        if r.status_code == 200:
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            size = os.path.getsize(filepath) / (1024**3)
            print(f"   Done — {size:.2f} GB")
        else:
            print(f"   Bad response: {r.status_code}")
    except Exception as e:
        print(f"   Failed: {e}")

print("NOAA download finished.")

   Done — 1.21 GB
   Done — 1.23 GB
   Done — 1.26 GB
   Done — 1.26 GB
   Done — 1.26 GB
All NOAA done!


In [ ]:
# Add more months to get closer to 20GB
more_urls = [
    "https://zenodo.org/record/7923702/files/flightlist_20190401_20190430.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190501_20190531.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190601_20190630.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190701_20190731.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190801_20190831.csv.gz",
    "https://zenodo.org/record/7923702/files/flightlist_20190901_20190930.csv.gz",
]

for url in more_urls:
    filename = url.split("/")[-1]
    filepath = f"/content/data/opensky/{filename}"
    print(f"Downloading {filename}...")
    try:
        r = requests.get(url, timeout=180)
        with open(filepath, "wb") as f:
            f.write(r.content)
        size = os.path.getsize(filepath) / (1024**2)
        print(f"   Done — {size:.1f} MB")
    except Exception as e:
        print(f"   Failed: {e}")

   Done — 158.3 MB
   Done — 169.5 MB
   Done — 177.3 MB
   Done — 193.6 MB
   Done — 199.9 MB
   Done — 191.9 MB


In [ ]:
import os

def get_folder_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

bts_size = get_folder_size("/content/data/bts")
noaa_size = get_folder_size("/content/data/noaa")
opensky_size = get_folder_size("/content/data/opensky")

total = bts_size + noaa_size + opensky_size

print(f" BTS Data:     {bts_size / (1024**3):.2f} GB")
print(f" NOAA Data:    {noaa_size / (1024**3):.2f} GB")
print(f" OpenSky Data: {opensky_size / (1024**3):.2f} GB")
print(f"")
print(f" TOTAL:        {total / (1024**3):.2f} GB")

 BTS Data:     13.30 GB
 NOAA Data:    6.23 GB
 OpenSky Data: 1.98 GB

 TOTAL:        21.51 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# mount google drive

from google.colab import drive
drive.mount('/content/drive')

import os
# Create SkyPredict folder in your Drive
os.makedirs("/content/drive/MyDrive/SkyPredict/processed", exist_ok=True)
os.makedirs("/content/drive/MyDrive/SkyPredict/models", exist_ok=True)
os.makedirs("/content/drive/MyDrive/SkyPredict/results", exist_ok=True)

print(" Google Drive mounted!")
print(" Folders created in your Drive under SkyPredict/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Google Drive mounted!
 Folders created in your Drive under SkyPredict/


In [ ]:
from pyspark.sql import functions as F

print("Loading BTS data into Spark...")
bts_df = spark.read.option("header", "true") \
               .option("inferSchema", "true") \
               .csv("/content/data/bts/*.csv")

# First check actual column names
print("Actual columns:", bts_df.columns)

Loading BTS data into Spark...
Actual columns: ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'Late

In [ ]:
from pyspark.sql import functions as F

# Clean & select using CORRECT column names
bts_clean = bts_df.select(
    "FlightDate",           # was FL_DATE
    "Reporting_Airline",    # was OP_CARRIER
    "Origin",               # same
    "Dest",                 # same
    "DepDelay",             # was DEP_DELAY
    "ArrDelay",             # was ARR_DELAY
    "Cancelled",            # same
    "WeatherDelay",         # was WEATHER_DELAY
    "CarrierDelay",         # was CARRIER_DELAY
    "Distance",             # same
    "DepTime",              # was DEP_TIME
    "ArrTime",              # was ARR_TIME
    "Month",                # extra - useful for ML
    "DayOfWeek",            # extra - useful for ML
    "Year"                  # extra - useful for ML
).filter(F.col("Cancelled") == 0) \
 .filter(F.col("DepDelay").isNotNull()) \
 .filter(F.col("ArrDelay").isNotNull())

print(f"Clean BTS rows: {bts_clean.count():,}")
bts_clean.show(5)
print(" BTS cleaned successfully!")

Clean BTS rows: 30,821,441
+----------+-----------------+------+----+--------+--------+---------+------------+------------+--------+-------+-------+-----+---------+----+
|FlightDate|Reporting_Airline|Origin|Dest|DepDelay|ArrDelay|Cancelled|WeatherDelay|CarrierDelay|Distance|DepTime|ArrTime|Month|DayOfWeek|Year|
+----------+-----------------+------+----+--------+--------+---------+------------+------------+--------+-------+-------+-----+---------+----+
|2019-01-04|               OO|   SBP| SFO|    -7.0|   -26.0|      0.0|        NULL|        NULL|   190.0|   1353|   1444|    1|        5|2019|
|2019-01-04|               OO|   IAH| XNA|    -5.0|     1.0|      0.0|        NULL|        NULL|   438.0|    930|   1119|    1|        5|2019|
|2019-01-04|               OO|   SGF| IAH|    -6.0|   -17.0|      0.0|        NULL|        NULL|   513.0|    637|    838|    1|        5|2019|
|2019-01-04|               OO|   ISN| DEN|   -21.0|   -29.0|      0.0|        NULL|        NULL|   576.0|   1314|  

In [ ]:
# Save BTS to personal Google Drive
bts_clean.write.mode("overwrite") \
    .parquet("/content/drive/MyDrive/SkyPredict/processed/bts_clean.parquet")

print(" BTS saved to Google Drive!")

 BTS saved to Google Drive!


In [ ]:
from pyspark.sql import functions as F

# Load NOAA data
print("Loading NOAA data into Spark...")
noaa_df = spark.read.option("header", "false") \
               .csv("/content/data/noaa/*.csv")

noaa_df = noaa_df.toDF("station_id", "date", "element",
                        "value", "m_flag", "q_flag", "s_flag", "obs_time")

noaa_clean = noaa_df.filter(
    F.col("element").isin(["PRCP", "SNOW", "TMAX", "TMIN", "AWND"])
).filter(F.col("q_flag").isNull()) \
 .filter(F.col("value").isNotNull())

print(f"Clean NOAA rows: {noaa_clean.count():,}")
noaa_clean.show(5)

# Save to Drive
noaa_clean.write.mode("overwrite") \
    .parquet("/content/drive/MyDrive/SkyPredict/processed/noaa_clean.parquet")

print(" NOAA saved to Google Drive!")

Loading NOAA data into Spark...
Clean NOAA rows: 126,666,117
+-----------+--------+-------+-----+------+------+------+--------+
| station_id|    date|element|value|m_flag|q_flag|s_flag|obs_time|
+-----------+--------+-------+-----+------+------+------+--------+
|CA006118240|20190101|   PRCP|   10|  NULL|  NULL|     C|    NULL|
|CA006118240|20190101|   SNOW|   10|  NULL|  NULL|     C|    NULL|
|CA006119055|20190101|   TMAX|  -10|  NULL|  NULL|     C|    NULL|
|CA006119055|20190101|   TMIN|  -65|  NULL|  NULL|     C|    NULL|
|CA006119055|20190101|   PRCP|    0|  NULL|  NULL|     C|    NULL|
+-----------+--------+-------+-----+------+------+------+--------+
only showing top 5 rows
 NOAA saved to Google Drive!


In [ ]:
from pyspark.sql import functions as F

# Load OpenSky data
print("Loading OpenSky data into Spark...")
opensky_df = spark.read.option("header", "true") \
                  .option("inferSchema", "true") \
                  .csv("/content/data/opensky/*.csv.gz")

print(f"Raw OpenSky rows: {opensky_df.count():,}")
print("Columns:", opensky_df.columns)
opensky_df.show(5)

# Save to Drive
opensky_df.write.mode("overwrite") \
    .parquet("/content/drive/MyDrive/SkyPredict/processed/opensky_clean.parquet")

print(" OpenSky saved to Google Drive!")

Loading OpenSky data into Spark...
Raw OpenSky rows: 30,307,666
Columns: ['callsign', 'number', 'icao24', 'registration', 'typecode', 'origin', 'destination', 'firstseen', 'lastseen', 'day', 'latitude_1', 'longitude_1', 'altitude_1', 'latitude_2', 'longitude_2', 'altitude_2']
+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+-------------------+-------------------+----------+------------------+------------------+----------+
|callsign|number|icao24|registration|typecode|origin|destination|          firstseen|           lastseen|                day|         latitude_1|        longitude_1|altitude_1|        latitude_2|       longitude_2|altitude_2|
+--------+------+------+------------+--------+------+-----------+-------------------+-------------------+-------------------+-------------------+-------------------+----------+------------------+------------------+----------+
|  CCA961| CA961|780c9e|      B-2006|    B77W

In [ ]:
import os

def get_folder_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total / (1024**3)  # GB

bts = get_folder_size('/content/drive/MyDrive/SkyPredict/processed/bts_clean.parquet')
noaa = get_folder_size('/content/drive/MyDrive/SkyPredict/processed/noaa_clean.parquet')
opensky = get_folder_size('/content/drive/MyDrive/SkyPredict/processed/opensky_clean.parquet')

print(f" BTS:     {bts:.2f} GB")
print(f" NOAA:    {noaa:.2f} GB")
print(f" OpenSky: {opensky:.2f} GB")
print(f"")
print(f" TOTAL:   {bts+noaa+opensky:.2f} GB")

 BTS:     0.23 GB
 NOAA:    0.45 GB
 OpenSky: 1.59 GB

 TOTAL:   2.27 GB


In [ ]:
print("=" * 40)
print("   SKY PREDICT — Dataset Summary")
print("=" * 40)
print(f"BTS Raw Data:        13.30 GB")
print(f"NOAA Raw Data:        6.23 GB")
print(f"OpenSky Raw Data:     1.98 GB")
print(f"")
print(f"TOTAL RAW:           21.51 GB   20GB+ requirement met!")
print(f"")
print(f"After Parquet compression:")
print(f"TOTAL PROCESSED:      2.27 GB")
print(f"")
print(f"Total Rows:         187,795,224 rows")
print("=" * 40)

   SKY PREDICT — Dataset Summary
BTS Raw Data:        13.30 GB
NOAA Raw Data:        6.23 GB
OpenSky Raw Data:     1.98 GB

TOTAL RAW:           21.51 GB   20GB+ requirement met!

After Parquet compression:
TOTAL PROCESSED:      2.27 GB

Total Rows:         187,795,224 rows
